# Day 8 | Lab 8.3: Microsoft Agent Framework (MAF) — Hands-On

**Duration:** ~1.5 hours

**Scenario.** Banking & financial services — preserved from the source AutoGen tutorial. Loan-application review, customer support escalation, and fraud investigation built as multi-agent teams using the open-source core of Microsoft Agent Framework.

**What is Microsoft Agent Framework (MAF)?** MAF is the unified evolution of two Microsoft agent stacks — **AutoGen** (now `autogen-agentchat` ≥ 0.4) and **Semantic Kernel** — integrated with **Azure AI Foundry** for managed deployment, identity, and observability. The `autogen-agentchat` package we install below **is** the open-source core of MAF; everything we build here runs unchanged inside MAF on Azure.

**Lineage at a glance.**

| Era | Package | What changed |
|---|---|---|
| Legacy | `pyautogen` (AutoGen 0.2) | Synchronous, monolithic, `ConversableAgent`-centric. **Forbidden in this curriculum.** |
| Open-source core (today) | `autogen-agentchat` ≥ 0.4 + `autogen-ext` | Async-first rewrite, layered API, model-client abstraction, `AssistantAgent` + team primitives. |
| Enterprise (today) | **Microsoft Agent Framework** on Azure AI Foundry | Adds managed identity (Entra ID), Azure OpenAI / Bedrock connectors, tracing into Application Insights, deployment as Azure Functions / Container Apps. |

**Learning Objectives.** By the end of this lab, you will be able to:
1. Set up `AssistantAgent`, `UserProxyAgent`, and `CodeExecutorAgent` with `OpenAIChatCompletionClient`.
2. Compose teams with `RoundRobinGroupChat` (deterministic) and `SelectorGroupChat` (LLM-picked next speaker).
3. Implement **Handoffs** for tiered support escalation.
4. Configure **AWS Bedrock** as the model backend via `AnthropicBedrockChatCompletionClient` (the path to running MAF on AWS-hosted Claude).
5. Build three banking workflows end-to-end: loan review, customer support escalation, fraud investigation.
6. Place MAF in the broader landscape via the **5-Framework Showdown** (LangGraph · OpenAI Agents SDK · CrewAI · MAF · Google ADK).

**Tools.** `autogen-agentchat>=0.4` · `autogen-ext` (OpenAI + Anthropic + Bedrock) · `gpt-4o` · `anthropic.claude-3-sonnet` (via Bedrock).

Created by Prashant Sahu · [LinkedIn](https://www.linkedin.com/in/prashantksahu/)*

---


## Migration from Legacy AutoGen 0.2

If you're coming from `pyautogen` (AutoGen 0.2 / AG2), three things change. **Imports** move from `autogen.ConversableAgent` to `autogen_agentchat.agents.AssistantAgent`. **Model config** moves from a `config_list` dict to a typed `OpenAIChatCompletionClient(model=...)` passed via `model_client=`. **Execution** is now `async`/`await` end-to-end (use `await team.run(task=...)` or stream via `Console(team.run_stream(...))`); the synchronous `agent.initiate_chat()` is gone.


---

## Section 1: Environment Setup

In [ ]:
# Required packages for this lab — already installed in your local venv.
# To install standalone, uncomment the line(s) below:
# !pip install -q 'autogen-agentchat>=0.4' 'autogen-ext[openai,anthropic]' boto3


### Verify Installation

In [ ]:
import importlib.metadata

# Define the packages we want to verify
packages = ['autogen-agentchat', 'autogen-core', 'autogen-ext']

# Loop through and print the version of each package to confirm installation
for pkg in packages:
    try:
        version = importlib.metadata.version(pkg)
        print(f"✅ {pkg}: {version}")
    except:
        print(f"❌ {pkg}: Not installed")

### API Key Configuration

In [ ]:
import os

# Local-venv pattern: load from .env if python-dotenv is available, otherwise rely on
# environment variables already set in your shell or venv activation script.
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

for key in ['OPENAI_API_KEY', 'AWS_ACCESS_KEY_ID', 'AWS_SECRET_ACCESS_KEY', 'AWS_DEFAULT_REGION']:
    status = '✅ Loaded' if os.environ.get(key) else '❌ MISSING'
    print(f'{key}: {status}')


### AWS Bedrock Credentials

Bedrock examples in §6 read AWS credentials from environment variables. Set them in your shell or `.env` file — never hardcode them in the notebook.


In [ ]:
import os

aws_access_key = os.environ.get('AWS_ACCESS_KEY_ID', '')
aws_secret_key = os.environ.get('AWS_SECRET_ACCESS_KEY', '')
aws_region = os.environ.get('AWS_DEFAULT_REGION', 'us-east-1')

if not (aws_access_key and aws_secret_key):
    print('⚠️  AWS credentials missing — §6 (Bedrock) cells will fail. '
          'Set AWS_ACCESS_KEY_ID and AWS_SECRET_ACCESS_KEY in your environment.')
else:
    print(f'✅ AWS credentials loaded for region: {aws_region}')


---

## Section 2: Core Concepts

---

## 3. Core Concepts

### 3.1 Agents

Agents are the building blocks of AutoGen. Each agent has:
- A **name** (unique identifier)
- A **system message** (defines personality and behavior)
- Optional **tools** (functions the agent can call)
- A **model client** (the LLM that powers the agent)

#### AssistantAgent

The most common agent type—powered by an LLM:

In [ ]:
from autogen_agentchat.agents import AssistantAgent
from autogen_ext.models.openai import OpenAIChatCompletionClient

# 1. Create a Model Client
# This object manages the connection to the LLM (GPT-4o in this case).
model_client = OpenAIChatCompletionClient(model="gpt-4o")

# 2. Create an Assistant Agent
# The AssistantAgent is an autonomous agent powered by the LLM.
# 'name': A unique identifier for the agent.
# 'system_message': Instructions that define the agent's persona and behavior.
assistant = AssistantAgent(
    name="Financial_Advisor",
    model_client=model_client,
    system_message="""You are a certified financial advisor.
    Help customers with investment questions.
    Always include appropriate disclaimers."""
)

#### UserProxyAgent

Allows human input during conversations:

In [ ]:
from autogen_agentchat.agents import UserProxyAgent

# Create a UserProxy Agent
# This agent acts as a bridge for human interaction.
# 'input_func=input': Configures the agent to prompt the user for text input via the notebook console.
user_proxy = UserProxyAgent(
    name="Customer",
    input_func=input
)

### 3.2 Teams

Teams define how agents collaborate. AutoGen provides several team types:

| Team Type | Description | Best For |
|-----------|-------------|----------|
| `RoundRobinGroupChat` | Agents take turns in fixed order | Simple workflows |
| `SelectorGroupChat` | LLM decides who speaks next | Complex, dynamic workflows |
| `Swarm` | Agents hand off to each other | Escalation patterns |

### 3.3 Termination Conditions

Define when a conversation should end:

In [ ]:
from autogen_agentchat.conditions import (
    TextMentionTermination,  # Stops conversation when specific text is generated
    MaxMessageTermination,   # Stops conversation after a certain number of messages
    TimeoutTermination,      # Stops conversation after a set time
)

# Example 1: End conversation if any agent types "APPROVED"
termination_approval = TextMentionTermination("APPROVED")

# Example 2: Complex Termination
# End if "DONE" is mentioned OR (|) if the message count exceeds 20.
# This prevents infinite loops while allowing for natural conclusion.
termination = TextMentionTermination("DONE") | MaxMessageTermination(20)

### 3.4 Tools

Tools are Python functions that agents can call:

In [ ]:
async def check_account_balance(account_id: str) -> dict:
    """
    Check the balance of a customer account.

    This is a 'tool' - a Python function that an agent can call.
    """
    # In a real app, this would call a banking API or database.
    return {"account_id": account_id, "balance": 15000.00}

# Create an Agent with Tools
# By passing 'tools=[check_account_balance]', we give the LLM the ability
# to decide when to call this function based on user queries.
assistant = AssistantAgent(
    name="Banking_Assistant",
    model_client=model_client,
    tools=[check_account_balance],
    system_message="You help customers check their accounts."
)

---

## Section 3: Multi-Agent Collaboration

---

## 4. Multi-Agent Collaboration

### 4.1 Two-Agent Chat

The simplest pattern: one assistant, one user.

```
┌─────────────────┐     message      ┌─────────────────┐
│  UserProxyAgent │ ───────────────► │ AssistantAgent  │
│   (Human/Auto)  │ ◄─────────────── │   (LLM-based)   │
└─────────────────┘     response     └─────────────────┘
```

**Example: Simple Banking Assistant**

In [11]:
import asyncio
from autogen_agentchat.agents import AssistantAgent, UserProxyAgent
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.conditions import TextMentionTermination
from autogen_agentchat.ui import Console
from autogen_ext.models.openai import OpenAIChatCompletionClient

async def simple_banking_chat():
    # 1. Setup the Model Client
    # We use GPT-4o for high-quality responses.
    model_client = OpenAIChatCompletionClient(model="gpt-4o")

    # 2. Create the Banking Assistant Agent
    # This agent behaves as the bank representative.
    assistant = AssistantAgent(
        name="Banking_Assistant",
        model_client=model_client,
        system_message="""You are a helpful bank customer service agent.
        Help customers with account questions, explain products,
        and guide them through banking processes.
        When the conversation is complete, say 'GOODBYE'."""
    )

    # 3. Create the Customer (User Proxy) Agent
    # 'input_func=input' enables interactive mode.
    # IMPORTANT: When you run this cell, look for a text input box in the output area to type your reply.
    user_proxy = UserProxyAgent(
        name="Customer",
        input_func=input
    )

    # 4. Define Termination Condition
    # The chat loop will terminate automatically when the assistant types "GOODBYE".
    termination = TextMentionTermination("GOODBYE")

    # 5. Create the Team
    # RoundRobinGroupChat dictates that agents speak in a strict sequence (Assistant -> User -> Assistant...)
    team = RoundRobinGroupChat(
        participants=[assistant, user_proxy],
        termination_condition=termination
    )

    # 6. Run the Conversation
    # Console() is a helper that pretty-prints the message stream to the notebook output.
    print("Starting chat... (Please check the output for an input box to type your response)")
    await Console(team.run_stream(
        task="Hi, I'd like to understand my savings account options."
    ))

# Execute the chat using top-level await
# This allows the async function to run within Colab's existing event loop.
await simple_banking_chat()

Starting chat... (Please check the output for an input box to type your response)
---------- TextMessage (user) ----------
Hi, I'd like to understand my savings account options.
---------- TextMessage (Banking_Assistant) ----------
Of course! I'd be happy to help you with that. Savings accounts are designed to help you save money and earn interest over time. Here are a few common types of savings accounts you might consider:

1. **Traditional Savings Account:** These accounts typically offer low interest rates but have easy access to your funds. They are a great starting point for basic savings needs.

2. **High-Yield Savings Account:** These accounts offer higher interest rates compared to traditional savings accounts. They are typically available through online banks and may have some restrictions on withdrawals.

3. **Money Market Account:** This type of savings account usually requires a higher minimum balance but offers higher interest rates. It may also come with check-writing pr

In [ ]:
# Run this cell if you want to restart the banking chat defined above.
# We use 'await' because simple_banking_chat is an asynchronous function.
await simple_banking_chat()

### 4.2 Multi-Agent Group Chat

Multiple specialized agents working together on complex tasks.

```
┌──────────────────────────────────────────────────────────┐
│                    SelectorGroupChat                      │
│              (LLM decides who speaks next)                │
└──────────────────────────────────────────────────────────┘
                           │
      ┌────────────────────┼────────────────────┐
      ▼                    ▼                    ▼
┌─────────────┐    ┌─────────────┐    ┌─────────────┐
│   Analyst   │    │ Compliance  │    │    Risk     │
│   Agent     │    │   Agent     │    │   Agent     │
└─────────────┘    └─────────────┘    └─────────────┘
```

**Example: Loan Application Review Team**


In [12]:
import asyncio
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.teams import SelectorGroupChat
from autogen_agentchat.conditions import TextMentionTermination, MaxMessageTermination
from autogen_agentchat.ui import Console
from autogen_ext.models.openai import OpenAIChatCompletionClient

async def loan_review_team():
    # Initialize the LLM client
    model_client = OpenAIChatCompletionClient(model="gpt-4o")

    # --- Define Specialized Agents ---

    # 1. Financial Analyst: Checks the numbers
    analyst = AssistantAgent(
        name="Financial_Analyst",
        model_client=model_client,
        system_message="""You are a senior financial analyst.
        Your responsibilities:
        - Analyze financial statements
        - Calculate debt-to-income ratios
        - Assess creditworthiness
        - Recommend loan amounts
        Always provide specific numbers and calculations.""",
        description="Analyzes financial data and creditworthiness"
    )

    # 2. Compliance Officer: Checks the rules
    compliance = AssistantAgent(
        name="Compliance_Officer",
        model_client=model_client,
        system_message="""You are a bank compliance officer.
        Your responsibilities:
        - Verify KYC (Know Your Customer) requirements
        - Check AML (Anti-Money Laundering) compliance
        - Ensure regulatory requirements are met
        - Flag any compliance concerns
        Reference specific regulations when applicable.""",
        description="Ensures regulatory compliance"
    )

    # 3. Risk Assessor: Checks the risk
    risk_assessor = AssistantAgent(
        name="Risk_Assessor",
        model_client=model_client,
        system_message="""You are a risk assessment specialist.
        Your responsibilities:
        - Evaluate loan default probability
        - Assess collateral adequacy
        - Consider market/economic risks
        - Provide risk rating (Low/Medium/High)
        Always justify your risk assessment.""",
        description="Evaluates and rates risk levels"
    )

    # 4. Loan Officer: Makes the final call
    loan_officer = AssistantAgent(
        name="Loan_Officer",
        model_client=model_client,
        system_message="""You are the senior loan officer.
        Your responsibilities:
        - Review all team inputs
        - Make final approval/rejection decision
        - Set loan terms and conditions
        - Document decision rationale
        Conclude with: "DECISION: APPROVED" or "DECISION: REJECTED"
        followed by your reasoning.""",
        description="Makes final loan decisions"
    )

    # --- Orchestration ---

    # Termination: Stop when a decision is made OR after 20 turns to prevent infinite loops.
    termination = TextMentionTermination("DECISION:") | MaxMessageTermination(20)

    # SelectorGroupChat: The LLM chooses the next speaker based on the conversation state.
    # The 'selector_prompt' guides the model on the ideal workflow order.
    team = SelectorGroupChat(
        participants=[analyst, compliance, risk_assessor, loan_officer],
        model_client=model_client,
        termination_condition=termination,
        selector_prompt="""Coordinate the loan review process:
        1. Start with Financial_Analyst for financial analysis
        2. Then Compliance_Officer for regulatory check
        3. Then Risk_Assessor for risk evaluation
        4. Finally Loan_Officer for decision
        Select the appropriate agent for the current stage."""
    )

    # Loan application data input
    application = """
    LOAN APPLICATION
    ================
    Applicant: ABC Manufacturing Inc.
    Loan Amount: $500,000
    Purpose: Equipment purchase and working capital

    Financial Data:
    - Annual Revenue: $2,000,000
    - Net Income: $150,000
    - Total Assets: $1,200,000
    - Total Liabilities: $600,000
    - Years in Business: 5
    - Credit Score: 720

    Collateral:
    - Equipment (appraised at $300,000)
    - Accounts Receivable ($200,000)

    Please process this application through all reviews.
    """

    # Run the workflow non-interactively (the agents talk to each other)
    await Console(team.run_stream(task=application))

# Execute using top-level await for proper async handling in notebooks
await loan_review_team()

---------- TextMessage (user) ----------

    LOAN APPLICATION
    Applicant: ABC Manufacturing Inc.
    Loan Amount: $500,000
    Purpose: Equipment purchase and working capital

    Financial Data:
    - Annual Revenue: $2,000,000
    - Net Income: $150,000
    - Total Assets: $1,200,000
    - Total Liabilities: $600,000
    - Years in Business: 5
    - Credit Score: 720

    Collateral:
    - Equipment (appraised at $300,000)
    - Accounts Receivable ($200,000)

    Please process this application through all reviews.
    
---------- TextMessage (Financial_Analyst) ----------
To process the loan application for ABC Manufacturing Inc., I will analyze the financial statements, calculate the debt-to-income ratio, assess creditworthiness, and provide a loan recommendation.

1. **Analyzing Financial Statements:**

   - **Revenue and Profit Margin:**
     - Annual Revenue: $2,000,000
     - Net Income: $150,000
     - Profit Margin = (Net Income / Revenue) x 100 = ($150,000 / $2,000,000)

CancelledError: 

### 4.3 Handoffs and Escalation

Agents can hand off to other agents or humans:

In [ ]:
from autogen_agentchat.base import Handoff

# Define a Tier 1 Agent with Escalation (Handoff) capabilities
# 'handoffs': A list of rules allowing this agent to transfer control to another agent.
tier1_agent = AssistantAgent(
    name="Tier1_Support",
    model_client=model_client,
    handoffs=[
        # If the Tier 1 agent determines a technical issue, it hands off to Tier 2
        Handoff(target="Tier2_Technical", message="Escalating technical issue"),
        # If the issue is a complaint, it hands off to the Supervisor
        Handoff(target="Supervisor", message="Escalating to supervisor"),
    ],
    system_message="""You are Tier 1 support.
    Handle basic questions. Escalate complex issues to:
    - Tier2_Technical for technical problems
    - Supervisor for complaints or complex issues"""
)

---

## 5. Task Automation

### 5.1 Tool-Enabled Agents

Agents can call Python functions to interact with external systems.

**Example: Banking Operations Agent**

In [ ]:
import asyncio
from typing import Dict, Any
from autogen_agentchat.agents import AssistantAgent
from autogen_ext.models.openai import OpenAIChatCompletionClient

# --- Define Tools (Async Functions) ---
# Tools are Python functions that agents can call to interact with the "world" (APIs, DBs, etc.)

async def check_account_balance(account_id: str) -> Dict[str, Any]:
    """
    Check the balance of a customer account.
    Args:
        account_id: The unique account identifier
    Returns:
        Account balance information
    """
    # Mock database of accounts
    mock_accounts = {
        "ACC001": {"balance": 15000.00, "currency": "USD", "type": "Checking"},
        "ACC002": {"balance": 50000.00, "currency": "USD", "type": "Savings"},
    }

    if account_id in mock_accounts:
        return {"status": "success", "account_id": account_id, **mock_accounts[account_id]}
    return {"status": "error", "message": "Account not found"}


async def transfer_funds(
    from_account: str,
    to_account: str,
    amount: float,
    description: str = ""
) -> Dict[str, Any]:
    """
    Transfer funds between accounts.
    """
    if amount <= 0:
        return {"status": "error", "message": "Amount must be positive"}

    # Simulate a business rule: Large transfers need approval
    if amount > 10000:
        return {
            "status": "pending_approval",
            "message": "Large transfers require manager approval",
            "reference": "TXN-2024-001"
        }

    return {
        "status": "success",
        "reference": "TXN-2024-002",
        "from_account": from_account,
        "to_account": to_account,
        "amount": amount
    }


async def check_loan_eligibility(
    customer_id: str,
    loan_amount: float,
    loan_type: str
) -> Dict[str, Any]:
    """Check if a customer is eligible for a loan."""
    return {
        "status": "success",
        "customer_id": customer_id,
        "eligible": True,
        "max_eligible_amount": loan_amount * 1.2,
        "suggested_rate": 7.5,
        "suggested_term_months": 60
    }


async def banking_assistant():
    model_client = OpenAIChatCompletionClient(model="gpt-4o")

    # Create assistant and register the tools
    # The 'tools' parameter allows the agent to execute the Python functions defined above.
    assistant = AssistantAgent(
        name="Banking_Assistant",
        model_client=model_client,
        tools=[
            check_account_balance,
            transfer_funds,
            check_loan_eligibility
        ],
        system_message="""You are an intelligent banking assistant.

        You can help customers with:
        - Checking account balances
        - Making transfers (confirm before executing)
        - Checking loan eligibility

        Be helpful, professional, and security-conscious.
        When the request is complete, say "TASK_COMPLETE".""",
        reflect_on_tool_use=True  # The LLM will analyze the tool output and summarize it for the user
    )

    # Run a complex task that requires multiple tool calls
    # The agent will plan: Check Balance -> Check Loan Eligibility -> Summarize
    result = await assistant.run(
        task="Customer ID: CUST001. Check balance for ACC001 and "
             "see if I'm eligible for a $25,000 personal loan."
    )

    # Print the final response generated by the assistant
    print(result.messages[-1].content)

# Execute using top-level await for Jupyter compatibility
await banking_assistant()

---

## Section 4: Running on AWS Bedrock


---

## 6. Integrating with GenAI Models

AutoGen supports multiple model providers through its extension system.

### 6.1 OpenAI

```python
from autogen_ext.models.openai import OpenAIChatCompletionClient

# Basic OpenAI client
model_client = OpenAIChatCompletionClient(
    model="gpt-4o",
    # api_key="sk-..."  # Or set OPENAI_API_KEY env var
)

# With custom parameters
model_client = OpenAIChatCompletionClient(
    model="gpt-4o",
    temperature=0.7,
    max_tokens=2000
)
```
### 6.2 Azure OpenAI

```python
from autogen_ext.models.openai import AzureOpenAIChatCompletionClient

azure_client = AzureOpenAIChatCompletionClient(
    model="gpt-4o",
    azure_endpoint="https://your-resource.openai.azure.com/",
    api_key="your-api-key",
    api_version="2024-02-01"
)
```

---

### 6.3 AWS Bedrock

**Method 1: Using Anthropic Extension (Recommended for Claude)**

In [ ]:
from autogen_ext.models.anthropic import (
    AnthropicBedrockChatCompletionClient,
    BedrockInfo
)
from autogen_core.models import ModelInfo

# Initialize a Client for AWS Bedrock (using Anthropic Claude)
# This setup connects AutoGen to models hosted on AWS infrastructure.
bedrock_client = AnthropicBedrockChatCompletionClient(
    model="anthropic.claude-3-sonnet-20240229-v1:0",
    temperature=0.1,
    model_info=ModelInfo(
        vision=False,
        function_calling=True,
        json_output=False,
        family="claude",
        structured_output=True
    ),
    bedrock_info=BedrockInfo(
        aws_access_key=aws_access_key,      # Variables defined in the credentials cell
        aws_secret_key=aws_secret_key,
        aws_region=aws_region,
        # aws_session_token="..."           # Uncomment if using temporary session credentials
    )
)

### 6.4 Complete Bedrock Example

In [ ]:
import asyncio
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.conditions import TextMentionTermination
from autogen_agentchat.ui import Console
from autogen_ext.models.anthropic import AnthropicBedrockChatCompletionClient, BedrockInfo
from autogen_core.models import ModelInfo

async def bedrock_banking_example():
    # 1. Setup Bedrock Client (Using Claude)
    # NOTE: This requires valid AWS credentials to be set in the environment or variables.
    bedrock_client = AnthropicBedrockChatCompletionClient(
        model="anthropic.claude-3-sonnet-20240229-v1:0",
        temperature=0.1,
        model_info=ModelInfo(
            vision=False,
            function_calling=True,
            json_output=False,
            family="claude",
            structured_output=True
        ),
        bedrock_info=BedrockInfo(
            aws_access_key=aws_access_key, # Assumes these variables are set earlier
            aws_secret_key=aws_secret_key,
            aws_region=aws_region
        )
    )

    # 2. Create Agents using the Bedrock Client
    customer_service = AssistantAgent(
        name="Customer_Service",
        model_client=bedrock_client,
        system_message="""You are a friendly bank customer service agent.
        Help customers with their questions.
        When complete, say "RESOLVED"."""
    )

    financial_advisor = AssistantAgent(
        name="Financial_Advisor",
        model_client=bedrock_client,
        system_message="""You are a certified financial advisor.
        Provide investment guidance with appropriate disclaimers.
        When complete, say "RESOLVED"."""
    )

    # 3. Run Team
    termination = TextMentionTermination("RESOLVED")
    team = RoundRobinGroupChat(
        participants=[customer_service, financial_advisor],
        termination_condition=termination
    )

    await Console(team.run_stream(
        task="I want to understand my retirement investment options. "
             "I'm 45 years old with $100,000 to invest."
    ))

# Execute using top-level await
# Uncomment the line below to run if AWS credentials are valid
# await bedrock_banking_example()

---

## Section 5: Banking Use Cases End-to-End

---

## 7. Banking Use Cases

### 7.1 Automated Loan Processing

A complete loan processing workflow with specialized agents.

In [13]:
import asyncio
from datetime import datetime
from typing import Dict, Any
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.teams import SelectorGroupChat
from autogen_agentchat.conditions import TextMentionTermination, MaxMessageTermination
from autogen_agentchat.ui import Console
from autogen_ext.models.openai import OpenAIChatCompletionClient

# === 1. DEFINE TOOLS ===

async def fetch_credit_report(customer_id: str) -> Dict[str, Any]:
    """Fetch customer credit report"""
    return {
        "customer_id": customer_id,
        "credit_score": 720,
        "payment_history": "Good",
        "credit_utilization": "25%",
        "derogatory_marks": 0
    }

async def verify_income(customer_id: str, stated_income: float) -> Dict[str, Any]:
    """Verify customer income"""
    return {
        "customer_id": customer_id,
        "stated_income": stated_income,
        "verified_income": 95000,
        "verification_status": "VERIFIED",
        "employer": "Tech Corp Inc."
    }

async def check_fraud_indicators(customer_id: str) -> Dict[str, Any]:
    """Check for fraud indicators"""
    return {
        "customer_id": customer_id,
        "fraud_score": 0.05,
        "flags": [],
        "recommendation": "PROCEED"
    }

async def generate_loan_decision(
    customer_id: str,
    loan_amount: float,
    credit_score: int,
    dti: float
) -> Dict[str, Any]:
    """Generate final loan decision based on inputs"""
    if credit_score >= 700 and dti <= 40:
        decision = "APPROVED"
        rate = 6.5
    elif credit_score >= 650 and dti <= 43:
        decision = "APPROVED_WITH_CONDITIONS"
        rate = 8.5
    else:
        decision = "DECLINED"
        rate = None

    return {
        "decision": decision,
        "interest_rate": rate,
        "decision_timestamp": datetime.now().isoformat()
    }

# === 2. DEFINE MULTI-AGENT SYSTEM ===

async def loan_processing_system():
    model_client = OpenAIChatCompletionClient(model="gpt-4o")

    # Specialized Agents
    # Each agent has a narrow system message and specific tools
    credit_analyst = AssistantAgent(
        name="Credit_Analyst",
        model_client=model_client,
        tools=[fetch_credit_report],
        system_message="""You are a credit analyst.
        Pull and analyze credit reports.
        Calculate debt-to-income ratios.
        Flag any concerns.""",
        description="Analyzes credit reports"
    )

    income_verifier = AssistantAgent(
        name="Income_Verifier",
        model_client=model_client,
        tools=[verify_income],
        system_message="""You verify customer income.
        Check employment stability.
        Report verification status.""",
        description="Verifies income"
    )

    fraud_specialist = AssistantAgent(
        name="Fraud_Specialist",
        model_client=model_client,
        tools=[check_fraud_indicators],
        system_message="""You detect fraud indicators.
        Analyze applications for suspicious patterns.
        Provide fraud risk assessment.""",
        description="Checks for fraud"
    )

    loan_officer = AssistantAgent(
        name="Loan_Officer",
        model_client=model_client,
        tools=[generate_loan_decision],
        system_message="""You are the senior loan officer.
        Review all reports and make the final decision.

        Announce: "FINAL_DECISION: [APPROVED/DECLINED] - [Summary]" """,
        description="Makes final decision"
    )

    termination = TextMentionTermination("FINAL_DECISION:") | MaxMessageTermination(15)

    # SelectorGroupChat
    # The prompt guides the orchestration flow: Analyst -> Verifier -> Fraud -> Decision
    team = SelectorGroupChat(
        participants=[credit_analyst, income_verifier, fraud_specialist, loan_officer],
        model_client=model_client,
        termination_condition=termination,
        selector_prompt="""Coordinate the loan review:
        1. Credit_Analyst first
        2. Income_Verifier second
        3. Fraud_Specialist third
        4. Loan_Officer last for decision"""
    )

    # Loan Application Data
    application = """
    LOAN APPLICATION
    Customer ID: CUST-2024-001
    Loan Amount: $50,000 Personal Loan
    Stated Annual Income: $95,000
    Current Monthly Debts: $1,500

    Please process through all verification steps.
    """

    await Console(team.run_stream(task=application))

# Execute using top-level await
await loan_processing_system()

---------- TextMessage (user) ----------

    LOAN APPLICATION
    Customer ID: CUST-2024-001
    Loan Amount: $50,000 Personal Loan
    Stated Annual Income: $95,000
    Current Monthly Debts: $1,500

    Please process through all verification steps.
    
---------- ToolCallRequestEvent (Credit_Analyst) ----------
[FunctionCall(id='call_8WjKj0VAF5CNDYXmtrEmLMak', arguments='{"customer_id":"CUST-2024-001"}', name='fetch_credit_report')]
---------- ToolCallExecutionEvent (Credit_Analyst) ----------
[FunctionExecutionResult(content="{'customer_id': 'CUST-2024-001', 'credit_score': 720, 'payment_history': 'Good', 'credit_utilization': '25%', 'derogatory_marks': 0}", name='fetch_credit_report', call_id='call_8WjKj0VAF5CNDYXmtrEmLMak', is_error=False)]
---------- ToolCallSummaryMessage (Credit_Analyst) ----------
{'customer_id': 'CUST-2024-001', 'credit_score': 720, 'payment_history': 'Good', 'credit_utilization': '25%', 'derogatory_marks': 0}
---------- ToolCallRequestEvent (Loan_Officer)

### 7.2 Customer Support Escalation

Tiered support with automatic handoffs.

In [ ]:
import asyncio
from autogen_agentchat.agents import AssistantAgent, UserProxyAgent
from autogen_agentchat.teams import SelectorGroupChat
from autogen_agentchat.conditions import TextMentionTermination
from autogen_agentchat.base import Handoff
from autogen_ext.models.openai import OpenAIChatCompletionClient

async def customer_support_system():
    model_client = OpenAIChatCompletionClient(model="gpt-4o")

    # --- Define Handoffs ---
    # Handoffs allow agents to transfer the conversation to others based on intent.

    # Tier 1 Support
    tier1 = AssistantAgent(
        name="Tier1_Support",
        model_client=model_client,
        handoffs=[
            Handoff(target="Tier2_Technical", message="Escalating technical issue"),
            Handoff(target="Tier2_Billing", message="Escalating billing issue"),
            Handoff(target="Supervisor", message="Escalating to supervisor")
        ],
        system_message="""You are Tier 1 support.

        Handle: Basic inquiries, password resets, simple questions

        Escalate to:
        - Tier2_Technical: App/online banking issues
        - Tier2_Billing: Fee disputes, statements
        - Supervisor: Complaints, complex issues

        If resolved, say "RESOLVED"."""
    )

    # Tier 2 Technical
    tier2_tech = AssistantAgent(
        name="Tier2_Technical",
        model_client=model_client,
        handoffs=[Handoff(target="Supervisor", message="Needs supervisor")],
        system_message="""You are Tier 2 technical support.
        Handle: Online banking, mobile app, security issues.
        Say "RESOLVED" when done."""
    )

    # Tier 2 Billing
    tier2_billing = AssistantAgent(
        name="Tier2_Billing",
        model_client=model_client,
        handoffs=[Handoff(target="Supervisor", message="Needs approval")],
        system_message="""You are Tier 2 billing specialist.
        Handle: Fee disputes (up to $100), statements, rates.
        Say "RESOLVED" when done."""
    )

    # Supervisor
    supervisor = AssistantAgent(
        name="Supervisor",
        model_client=model_client,
        system_message="""You are a customer service supervisor.

        You can:
        - Waive fees up to $500
        - Offer promotional rates
        - Handle complaints

        Say "RESOLVED_BY_SUPERVISOR" when done."""
    )

    termination = (
        TextMentionTermination("RESOLVED") |
        TextMentionTermination("RESOLVED_BY_SUPERVISOR")
    )

    # Team: The SelectorGroupChat routes messages. The Handoffs defined above provide strict routing rules.
    team = SelectorGroupChat(
        participants=[tier1, tier2_tech, tier2_billing, supervisor],
        model_client=model_client,
        termination_condition=termination
    )

    # Test Case: An overdraft fee dispute should go Tier 1 -> Billing -> Supervisor
    await Console(team.run_stream(
        task="I was charged a $35 overdraft fee but I had money in my account!"
    ))

# Execute using top-level await
await customer_support_system()

### 7.3 Fraud Investigation

Multi-agent fraud detection and investigation.

In [ ]:
import asyncio
from typing import Dict
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.teams import SelectorGroupChat
from autogen_agentchat.conditions import TextMentionTermination
from autogen_ext.models.openai import OpenAIChatCompletionClient

# --- Fraud Detection Tools ---

async def analyze_transactions(account_id: str) -> Dict:
    """Analyze transaction history for anomalies."""
    return {
        "account_id": account_id,
        "anomalies": [
            {"type": "unusual_location", "risk": "HIGH"},
            {"type": "velocity", "details": "15 transactions in 1 hour", "risk": "MEDIUM"}
        ],
        "risk_score": 0.85
    }

async def check_device(session_id: str) -> Dict:
    """Check device and session security details."""
    return {
        "device_known": False,
        "ip_location": "Lagos, Nigeria",
        "customer_location": "New York, USA",
        "vpn_detected": True
    }

async def initiate_hold(account_id: str, reason: str) -> Dict:
    """Place a temporary hold on the account."""
    return {
        "hold_status": "ACTIVE",
        "reference": "HOLD-2024-001"
    }

async def fraud_investigation():
    model_client = OpenAIChatCompletionClient(model="gpt-4o")

    # Transaction Monitor Agent
    transaction_monitor = AssistantAgent(
        name="Transaction_Monitor",
        model_client=model_client,
        tools=[analyze_transactions],
        system_message="Analyze transaction patterns for anomalies.",
        description="Monitors transactions"
    )

    # Session Analyst Agent
    session_analyst = AssistantAgent(
        name="Session_Analyst",
        model_client=model_client,
        tools=[check_device],
        system_message="Analyze device/session security.",
        description="Analyzes sessions"
    )

    # Fraud Investigator Agent (Decision Maker)
    fraud_investigator = AssistantAgent(
        name="Fraud_Investigator",
        model_client=model_client,
        tools=[initiate_hold],
        system_message="""Review all analysis and decide:
        - FALSE_POSITIVE
        - SUSPICIOUS
        - LIKELY_FRAUD (place hold)
        - CONFIRMED_FRAUD

        Say: "FRAUD_DECISION: [TYPE] - [Reasoning]" """,
        description="Makes fraud determination"
    )

    termination = TextMentionTermination("FRAUD_DECISION:")

    # SelectorGroupChat
    # The model decides the best order of operations based on the agent descriptions.
    team = SelectorGroupChat(
        participants=[transaction_monitor, session_analyst, fraud_investigator],
        model_client=model_client,
        termination_condition=termination
    )

    alert = """
    FRAUD ALERT - HIGH PRIORITY
    Account: ACC-2024-5678
    Session: SES-9999-ABCD

    Activity: $5,000 wire + $2,500 ATM + 15 purchases ($3,200)
    All within 2 hours from unusual location.

    Please investigate.
    """

    await Console(team.run_stream(task=alert))

# Execute using top-level await
await fraud_investigation()

---

## Section 6: Best Practices & Quick Reference

---

## 8. Best Practices

### Security & Compliance

| Practice | Description |
|----------|-------------|
| **Human Approval** | Always require human approval for financial decisions |
| **Audit Logging** | Log all agent conversations for compliance |
| **PII Protection** | Never log or expose sensitive customer data |
| **Access Control** | Limit agent capabilities based on role |

### Error Handling

```python
from autogen_agentchat.conditions import MaxMessageTermination

# Always set maximum messages to prevent infinite loops
termination = (
    TextMentionTermination("DONE") |
    MaxMessageTermination(20)  # Safety limit
)

# Handle exceptions gracefully
try:
    result = await team.run(task=task)
except Exception as e:
    logger.error(f"Agent workflow failed: {e}")
    # Fallback to human handling
```

### Cost Management

```python
# Use cheaper models for simple tasks
simple_client = OpenAIChatCompletionClient(model="gpt-4o-mini")
advanced_client = OpenAIChatCompletionClient(model="gpt-4o")

# Tier 1 support: cheaper model
tier1 = AssistantAgent(name="Tier1", model_client=simple_client, ...)

# Complex decisions: advanced model
decision_maker = AssistantAgent(name="DecisionMaker", model_client=advanced_client, ...)
```

### Production Recommendations

1. **Use Docker for code execution** - Sandbox any code execution
2. **Implement timeouts** - Prevent runaway conversations
3. **Monitor costs** - Track token usage per agent
4. **Test thoroughly** - Unit test agents individually, integration test teams
5. **Version control prompts** - System messages are critical, version them
6. **Implement fallbacks** - Human escalation for edge cases



---

## Quick Reference

### Agent Types

| Agent | Purpose | Key Parameters |
|-------|---------|----------------|
| `AssistantAgent` | LLM-powered agent | `model_client`, `tools`, `system_message` |
| `UserProxyAgent` | Human input | `input_func` |
| `CodeExecutorAgent` | Run code | `code_executor` |

### Team Types

| Team | Orchestration | Best For |
|------|---------------|----------|
| `RoundRobinGroupChat` | Fixed turn order | Simple workflows |
| `SelectorGroupChat` | LLM selects next | Complex, dynamic |
| `Swarm` | Agent handoffs | Escalation paths |

### Termination Conditions

```python
TextMentionTermination("DONE")      # End when text appears
MaxMessageTermination(20)            # End after N messages
TimeoutTermination(300)              # End after 5 minutes

# Combine with | (or) and & (and)
combined = TextMentionTermination("DONE") | MaxMessageTermination(20)
```


---

## Section 7: The 5-Framework Showdown

We've now seen Microsoft Agent Framework hands-on (§§ 1-6). To choose between MAF and the alternatives in your day-to-day work, use the comparison below. The other 4 frameworks each get their own lab in this curriculum (LangGraph in 8.1/8.2, OpenAI Agents SDK in 8.5, CrewAI in 8.4, Google ADK in Module 9 reference) — this section gives you the side-by-side.


| Dimension | LangGraph | OpenAI Agents SDK | CrewAI | **MAF** (autogen-agentchat 0.4) | Google ADK |
|-----------|-----------|-------------------|--------|---------------|------------|
| **Philosophy** | Build the workflow graph | Define agents, let them handoff | Hire a team with roles | Agents debate in group chat | Agents in Google Cloud |
| **Orchestration** | Graph (nodes + edges) | Handoff-based | Role-based (seq/hierarchical) | Conversation-based | Agent-based |
| **Control Level** | Maximum — every path explicit | Medium — LLM decides handoffs | Medium — process type decides | Minimal — agents self-organize | Medium |
| **Agent Definition** | `create_agent(tools, system_prompt)` | `Agent(name, instructions, tools)` | `Agent(role, goal, backstory)` | `AssistantAgent(name, model_client, system_message)` | `Agent(model, tools)` |
| **Multi-Agent** | Subgraphs, Supervisor, Router | Handoffs, Agents-as-Tools | Sequential, Hierarchical | RoundRobin, Selector GroupChat | Multi-agent sessions |
| **State Mgmt** | Explicit TypedDict | Conversation history | Task context chain | Chat message history | Session state |
| **Tools** | `@tool` + LangChain ecosystem | `@function_tool` (auto schema) | `@tool` (CrewAI native) | Python functions + type hints | Google Cloud tools |
| **Guardrails** | Custom / NeMo / External | Built-in Input/Output | Callbacks, validation | Hooks | Built-in safety |
| **Memory** | MemorySaver + InMemoryStore | SQLiteSession | Crew memory | Chat history | Session persistence |
| **HITL** | `interrupt()` + `Command(resume=)` | Custom (not built-in) | Callbacks | Human agent in chat | Custom |
| **Debugging** | Checkpoints + LangSmith | OpenAI Dashboard traces | Verbose logs | Chat transcript logs | Cloud Trace |
| **Voice** | No | Yes (RealtimeAgent) | No | No | No |
| **MCP Support** | Yes | Yes | No | Yes | Yes |
| **LLM Providers** | Any (OpenAI, Anthropic, Groq) | OpenAI only | Any (via LiteLLM) | Any (via model clients) | Google (Gemini) |
| **Production Ready** | Highest | High | Medium | Low-Medium | High |
| **Best For** | Complex enterprise, compliance | OpenAI ecosystem, voice, quick | Content teams, creative | Research, debate, brainstorming | Google Cloud native |
| **Covered In** | Labs 8.1, 8.2 | Lab 8.5 | Lab 8.4 | Lab 8.3 (this lab) | Module 9 reference |

## When to Use Which Framework

**Decision Flowchart:**

1. **Need deterministic, auditable routing?** (compliance, banking, healthcare) → **LangGraph**
2. **Building on OpenAI only? Need voice?** (chatbots, IVR, customer service) → **OpenAI Agents SDK**
3. **Content creation? Role-based team?** (marketing, writing, research) → **CrewAI**
4. **Need agents to debate/brainstorm?** (analysis, research, strategy) → **AutoGen**
5. **Google Cloud ecosystem?** (GCP workloads, Gemini models) → **Google ADK**

**Rule of Thumb:** Start with the simplest framework that meets your needs. You can always add complexity later.

---

## Section 8: MAF on Azure — When Financial Clients Should Choose It Over LangGraph

Everything in §§ 1-6 runs on the open-source `autogen-agentchat` core. Microsoft Agent Framework *on Azure AI Foundry* adds enterprise infrastructure on top of that core. For a financial-services client, MAF on Azure is the right choice over LangGraph when **all** of the following are true:

1. **Azure-native estate.** The client already runs on Azure (AKS, Functions, Storage, AI Foundry). Identity flows through **Entra ID**; observability lands in **Application Insights**. MAF is first-class in this stack — LangGraph requires more glue.
2. **Governed model access.** The client cannot expose data to public OpenAI; they consume **Azure OpenAI** with private endpoints, content filters, and quota controls. MAF speaks Azure OpenAI natively; LangGraph supports it but as one of many providers.
3. **Conversation-shaped problem.** The workflow is a *team that talks* (loan review committee, fraud investigation panel, compliance debate) rather than a *workflow that branches*. Group-chat patterns are MAF's home turf; LangGraph's strength is explicit DAGs.
4. **Need managed deployment + lifecycle.** MAF agents deploy as Azure Container Apps / Functions with built-in versioning, A/B routing, and rollback. LangGraph requires you to wrap and host the graph yourself (or use LangGraph Platform — a separate vendor).
5. **Microsoft procurement is already in place.** Practical reality at large banks: an existing Microsoft EA radically shortens vendor onboarding vs adding a new SaaS (LangSmith / LangGraph Platform).

**Stay on LangGraph instead** when the workflow is a strict, audited graph (compliance reporting as in Lab 8.2), when you need cross-cloud portability, or when explicit-state/HITL semantics are primary. **Stay on MAF (Azure or open-source)** when the workflow is conversation-driven and the agents need to negotiate.


---
## 9. Conclusion & Key Takeaways

### What We Covered

| Concept | Takeaway |
|---|---|
| **MAF lineage** | AutoGen 0.2 (`pyautogen`) → autogen-agentchat 0.4 (open-source core, async, layered) → MAF on Azure AI Foundry (managed). |
| **`AssistantAgent`** | LLM-powered agent. `name` + `model_client` + `system_message` (+ optional `tools`, `handoffs`). |
| **`RoundRobinGroupChat`** | Fixed turn order. Deterministic, cheap to reason about, ideal when each agent has a clear hand-off cue. |
| **`SelectorGroupChat`** | An LLM picks the next speaker each turn. Better when the conversation isn't a pipeline (debate, investigation). |
| **`Handoff`** | Declarative escalation. Each `Handoff(target=..., message=...)` becomes an LLM-callable tool that transfers control. |
| **Termination conditions** | `TextMentionTermination(...) | MaxMessageTermination(...)` — always set both; loops without a cap are a production-incident waiting to happen. |
| **Bedrock via `AnthropicBedrockChatCompletionClient`** | Same agent code, different `model_client`. The path to running MAF on AWS-hosted Claude. |
| **Async-first** | All MAF agent calls are `async`/`await`. Wrap in `asyncio.run(...)` or `await` inside an async function — no synchronous fallback. |
| **MAF vs LangGraph** | Conversation-shaped workflows + Azure estate → MAF. Strict audited DAGs + cross-cloud → LangGraph. |

**Next Lab:** Lab 8.4 — CrewAI: Role-Based Multi-Agent for Fintech Content 🧑‍💼


## 10. Stretch Exercise (Optional)

1. Convert the loan-review team (§3) from `SelectorGroupChat` to `RoundRobinGroupChat`. Compare conversation flow and final-decision quality on the same input.
2. Add a 4th specialist (e.g., **Legal Reviewer**) to the loan-review team. Update the selector prompt and re-run. Does the LLM correctly route legal questions to the new agent?
3. Wire `Handoff` into the loan-review team so the Compliance Officer can hand off to a Legal Reviewer when AML rules conflict with state regulation. Test with a deliberately ambiguous case.
4. Replace the OpenAI model client with `AnthropicBedrockChatCompletionClient` (Claude Sonnet via Bedrock) for §3's loan-review team. Verify zero code changes outside the model-client constructor.
5. Add a custom termination condition: stop when the Risk Officer's message contains a JSON object with `decision: 'APPROVED' | 'REJECTED'`. (Hint: subclass `TerminationCondition`.)
6. Instrument every agent call with OpenTelemetry spans (use `opentelemetry-instrumentation-openai`). Inspect the trace in Jaeger or Application Insights — preview of MAF-on-Azure observability.


---

## Interview Preparation

The questions below mirror what client interviewers commonly ask about the topics in this lab. Use the hint to think through the answer first; use the sketch only to verify your reasoning.

---

**Q1. What is the lineage AutoGen 0.2 → autogen-agentchat 0.4 → MAF — what changed at each step?**

*Hint:* Three eras, three step changes.

*Answer sketch:* **0.2 (`pyautogen`):** synchronous, monolithic, `ConversableAgent` did everything. **0.4 (`autogen-agentchat`):** ground-up async rewrite, layered API (core / agentchat / ext), model-client abstraction so agents are model-agnostic, group-chat primitives (RoundRobin, Selector, Swarm). **MAF on Azure:** wraps 0.4 with managed identity (Entra ID), Azure OpenAI / Bedrock connectors, Application Insights tracing, deployment as Azure Container Apps / Functions. The Python code from 0.4 runs unchanged inside MAF.

---

**Q2. MAF vs LangGraph — what's the core architectural difference?**

*Hint:* Conversations vs graphs.

*Answer sketch:* MAF models multi-agent as a **conversation**: agents speak in turn (RoundRobin / Selector), state is the chat transcript, handoffs are messages. LangGraph models multi-agent as a **state graph**: agents are nodes, state is an explicit TypedDict, handoffs are edges via `Command(goto=...)`. MAF is more flexible for fluid debate; LangGraph is more auditable for strict workflows. Both can do everything the other does — the question is which paradigm matches the problem.

---

**Q3. What is `RoundRobinGroupChat` vs `SelectorGroupChat`?**

*Hint:* Who decides the next speaker.

*Answer sketch:* **`RoundRobinGroupChat`:** speakers cycle in fixed insertion order. Deterministic, debuggable, ideal when each agent has a clear handoff cue (Coder → Tester → Reviewer). **`SelectorGroupChat`:** an LLM looks at the conversation so far and picks the most relevant next speaker from a pool. Better when the conversation isn't a pipeline (loan committee, fraud investigation, brainstorm). Selector adds an LLM call per turn — pay for flexibility.

---

**Q4. When does MAF make sense over LangGraph for a financial-services client?**

*Hint:* Three signals: Azure estate + conversation shape + Microsoft procurement.

*Answer sketch:* Pick MAF when (a) the client is Azure-native and wants Entra ID + Application Insights + Azure OpenAI out of the box, (b) the workflow is conversation-shaped (committee debate, group chat) rather than a strict DAG, and (c) Microsoft procurement is already in place. Pick LangGraph when the workflow is an audited DAG (compliance reporting), when you need explicit state/HITL semantics, or when cross-cloud portability matters. Both ship to production; the choice is fit-for-purpose.

---

**Q5. How does `Handoff` work — what's it underneath?**

*Hint:* It's a declarative tool the agent can call.

*Answer sketch:* `Handoff(target='Tier2_Technical', message='Escalating')` is registered with the `AssistantAgent` and surfaces to the LLM as a callable tool. When the LLM decides to escalate, it 'calls' the handoff; the runtime intercepts, transfers control to the named target agent, and passes the handoff message along. So mechanically: function-calling under the hood, conversation-flow on the surface. Equivalent to LangGraph's `Command(goto=...)` but expressed via the tool-call protocol instead of a Python return value.

---

**Q6. What does Azure AI Foundry add that the open-source autogen-agentchat doesn't?**

*Hint:* Five things: identity, deployment, models, observability, governance.

*Answer sketch:* (1) **Entra ID** for agent identity and tool permissions — no API keys in code. (2) **Managed deployment** as Azure Container Apps / Functions with versioning + rollback. (3) **Azure OpenAI / Foundry Models** with private endpoints, content filters, quota. (4) **Application Insights** tracing — every agent turn becomes a span automatically. (5) **Foundry Governance** — content filters, Responsible AI dashboards, policy enforcement at the platform level. Without Foundry, you build all of this yourself.

---

**Q7. How would you instrument MAF for production observability?**

*Hint:* OpenTelemetry today; Application Insights on Azure tomorrow.

*Answer sketch:* On the open-source core: wrap each agent's model-client with OpenTelemetry — `opentelemetry-instrumentation-openai` auto-spans every chat completion. Add custom spans around `team.run(...)` to capture turn boundaries. Export to Jaeger/Tempo. On MAF on Azure: Foundry auto-instruments to Application Insights — every agent turn, tool call, and handoff appears as a structured span. Either way: log the full message transcript per run for replay; cap message size to avoid PII leakage in logs.

---

**Q8. What's the Python-async story in MAF — how is it different from LangGraph?**

*Hint:* MAF is async-only; LangGraph offers both.

*Answer sketch:* MAF: every agent call is `async` — `await team.run(task=...)` or stream via `async for msg in team.run_stream(...)`. There's no synchronous fallback; calling from sync code requires `asyncio.run(...)`. LangGraph: graphs are runnable both sync (`graph.invoke(...)`) and async (`await graph.ainvoke(...)`); each node can be either. In practice MAF's async-only choice is fine for service code (FastAPI, Azure Functions) but inconvenient for notebooks — that's why every example here is wrapped in an `async def` + `await`.



# LangGraph vs CrewAI vs Microsoft Agent Framework (MAF)

| Dimension | **LangGraph** | **CrewAI** | **Microsoft Agent Framework (MAF)** |
|---|---|---|---|
| **Maintainer** | LangChain Inc. | CrewAI Inc. (independent) | Microsoft |
| **Maturity / Release** | Stable 1.0; mature, widely adopted | 44.6K+ GitHub stars, production-ready | 1.0 GA on April 3, 2026 — unifies AutoGen + Semantic Kernel into a single product |
| **Core Paradigm** | Graph-based state machine — state flows through nodes and edges | Role-based teams ("crew") of role-playing agents with role, goal, backstory | Unified runtime for agentic workflows; YAML-declarative agent definitions + code |
| **Mental Model** | "Nodes and edges, with explicit state transitions" | "Cast of human-like agents on a team" | "Enterprise agent runtime spanning .NET and Python" |
| **Languages** | Python, TypeScript | Python only | .NET and Python |
| **Multi-Agent Orchestration** | Explicit via graph topology; supervisor / hierarchical / cyclic | Sequential, hierarchical (manager delegates), consensual (agents vote) | Group chat, handoff, sequential, custom orchestration (inherited from AutoGen + SK patterns) |
| **State Management** | Built-in checkpointing, durable execution, fine-grained state schema | No built-in checkpointing; coarse error handling; communication mediated through task outputs | Native persistence + thread/state management with Azure integration |
| **Human-in-the-Loop** | First-class (interrupt / resume on graph nodes) | Limited / via task design | Built-in approval and intervention patterns |
| **Model Support** | Model-agnostic (any LLM provider) | Model-agnostic | Model-agnostic; deep tie-ins to Azure OpenAI / Foundry |
| **Protocol Support (MCP / A2A)** | MCP via LangChain ecosystem; A2A not native | A2A support added; MCP first-class | A2A, MCP, and AG-UI supported out of the box |
| **Observability** | LangSmith (trace-level visibility per node) | Built-in logs + integrates with LangFuse, AgentOps | Azure Monitor, Application Insights, OpenTelemetry |
| **Cloud / Enterprise Fit** | Cloud-agnostic; LangGraph Platform for hosted deployment | Cloud-agnostic | Best fit for .NET shops, deep Azure integration, Foundry Agent Service for managed hosting |
| **Learning Curve** | Steepest — verbose, requires state schema, nodes, edges, compilation | Easiest — prototype in ~20 lines | Moderate; YAML-first lowers code overhead but ecosystem is broad |
| **Control vs. Speed** | Most control, lowest speed-to-prototype | Fastest to prototype; less fine-grained control | Balanced; declarative for simple cases, code for complex |
| **Best For** | Complex, stateful, branching workflows; production systems needing fault tolerance, retries, and audit trails | Role-based collaboration (Researcher → Analyst → Writer); rapid prototyping; business workflow automation | Enterprise/regulated environments, M365 + Copilot integration, mixed .NET/Python teams, Azure-native deployments |
| **Weak Spots** | Verbose for simple flows; learning curve | Limited at scale; teams often migrate to LangGraph when they need production-grade state management | Younger as a unified product (post-merger churn); strongest value tied to Microsoft stack |
| **License** | MIT (open source) | MIT (open source) | MIT (open source) |

### Picking one for BFSI work
- **Auditable, regulated workflows with retries + HIL approvals** (e.g., dispute routing, KYC remediation, loan adjudication) → **LangGraph**
- **Role-based research/analysis crews** (e.g., credit memo drafting: Researcher → Risk Analyst → Reviewer) → **CrewAI**
- **Bank already on Azure / .NET / Microsoft 365** with compliance needs and Foundry tooling → **MAF**

A common production pattern is to **prototype in CrewAI**, then migrate the critical paths to **LangGraph** (or **MAF** if Azure-native) once state, observability, and recovery requirements harden.